In [2]:
import SimpleITK as sitk
import os
import json


def volume_info(path):
    img = sitk.ReadImage(path)
    sz  = img.GetSize()
    sp  = img.GetSpacing()
    ori = img.GetOrigin()
    z_extent_mm = sz[2] * sp[2]

    return {
        "file": os.path.basename(path),
        "slices": sz[2],
        "spacing_z_mm": round(sp[2], 2),
        "z_extent_mm": round(z_extent_mm, 1),
        "z_start_mm": round(ori[2], 1),
        "z_end_mm": round(ori[2] + z_extent_mm, 1),
    }


def calling_function(input_dir: str, file_postfix: str, output_json: str = "volume_info_results.json"):
    results = {}

    for study_id in os.listdir(input_dir):
        study_path = os.path.join(input_dir, study_id)
        if not os.path.isdir(study_path):
            continue

        results[study_id] = {}

        for file_name in os.listdir(study_path):
            if not file_name.endswith(file_postfix):
                continue

            file_path = os.path.join(study_path, file_name)
            series_name = file_name.replace(file_postfix, "")
            results[study_id][series_name] = volume_info(file_path)

    with open(output_json, "w") as f:
        json.dump(results, f, indent=2)

    print(f"Results saved to {output_json}")
    return results

In [5]:
calling_function(input_dir="../ncct_cect/vindr_ds/baseline_volumes", file_postfix="_baseline.nii.gz", output_json="baseline_volume_info.json")

Results saved to baseline_volume_info.json


{'1.2.840.113619.2.278.3.717616.286.1587944967.557': {'1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4198401': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4198401_baseline.nii.gz',
   'slices': 205,
   'spacing_z_mm': 1.5,
   'z_extent_mm': 307.5,
   'z_start_mm': -163.2,
   'z_end_mm': 144.2},
  '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4202498': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4202498_baseline.nii.gz',
   'slices': 205,
   'spacing_z_mm': 1.5,
   'z_extent_mm': 307.5,
   'z_start_mm': -163.2,
   'z_end_mm': 144.2},
  '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.562.3': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.562.3_baseline.nii.

In [6]:
calling_function(input_dir="../ncct_cect/vindr_ds/aligned_volumes", file_postfix="_aligned.nii.gz", output_json="aligned_volume_info.json")

Results saved to aligned_volume_info.json


{'1.2.840.113619.2.278.3.717616.286.1587944967.557': {'1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4202498': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4202498_aligned.nii.gz',
   'slices': 292,
   'spacing_z_mm': 1.5,
   'z_extent_mm': 438.0,
   'z_start_mm': -439.2,
   'z_end_mm': -1.2},
  '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.562.3': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.562.3_aligned.nii.gz',
   'slices': 292,
   'spacing_z_mm': 1.5,
   'z_extent_mm': 438.0,
   'z_start_mm': -439.2,
   'z_end_mm': -1.2},
  '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4198401': {'file': '1.2.840.113619.2.278.3.717616.286.1587944967.557_1.2.840.113619.2.278.3.717616.286.1587944967.658.4198401_aligned.nii.gz',


## extract seg masks

In [ ]:
"""
extract_mask_labels.py
----------------------
Extract unique voxel labels from a segmentation NIfTI/numpy file
and map them to organ names using the provided label_map.

Usage:
    python extract_mask_labels.py --seg path/to/segmentation.nii.gz
    python extract_mask_labels.py --seg path/to/mask.npy
    python extract_mask_labels.py --seg path/to/mask.npy --tier 2   # filter by tier
"""

import argparse
import json
import numpy as np
from pathlib import Path

# ── Label map (keys are integer label IDs) ──────────────────────────────────
LABEL_MAP = {
    1:   {"stem": "liver",                     "tier": 2, "in_reg": True},
    2:   {"stem": "spleen",                    "tier": 2, "in_reg": True},
    3:   {"stem": "kidney_left",               "tier": 2, "in_reg": True},
    4:   {"stem": "kidney_right",              "tier": 2, "in_reg": True},
    5:   {"stem": "pancreas",                  "tier": 2, "in_reg": True},
    6:   {"stem": "gallbladder",               "tier": 2, "in_reg": True},
    7:   {"stem": "autochthon_left",           "tier": 2, "in_reg": True},
    8:   {"stem": "autochthon_right",          "tier": 2, "in_reg": True},
    9:   {"stem": "iliopsoas_left",            "tier": 2, "in_reg": True},
    10:  {"stem": "iliopsoas_right",           "tier": 2, "in_reg": True},
    13:  {"stem": "aorta",                     "tier": 2, "in_reg": True},
    14:  {"stem": "inferior_vena_cava",        "tier": 2, "in_reg": True},
    15:  {"stem": "portal_vein_and_splenic_vein","tier": 2, "in_reg": True},
    16:  {"stem": "iliac_artery_left",         "tier": 2, "in_reg": True},
    17:  {"stem": "iliac_artery_right",        "tier": 2, "in_reg": True},
    18:  {"stem": "iliac_vena_left",           "tier": 2, "in_reg": True},
    19:  {"stem": "iliac_vena_right",          "tier": 2, "in_reg": True},
    21:  {"stem": "vertebrae_L1",              "tier": 1, "in_reg": True},
    22:  {"stem": "vertebrae_L2",              "tier": 1, "in_reg": True},
    23:  {"stem": "vertebrae_L3",              "tier": 1, "in_reg": True},
    24:  {"stem": "vertebrae_L4",              "tier": 1, "in_reg": True},
    25:  {"stem": "vertebrae_L5",              "tier": 1, "in_reg": True},
    26:  {"stem": "vertebrae_T10",             "tier": 1, "in_reg": True},
    27:  {"stem": "vertebrae_T11",             "tier": 1, "in_reg": True},
    28:  {"stem": "vertebrae_T12",             "tier": 1, "in_reg": True},
    29:  {"stem": "vertebrae_T1",              "tier": 1, "in_reg": True},
    30:  {"stem": "vertebrae_T2",              "tier": 1, "in_reg": True},
    31:  {"stem": "vertebrae_T3",              "tier": 1, "in_reg": True},
    32:  {"stem": "vertebrae_T4",              "tier": 1, "in_reg": True},
    33:  {"stem": "vertebrae_T5",              "tier": 1, "in_reg": True},
    34:  {"stem": "vertebrae_T6",              "tier": 1, "in_reg": True},
    35:  {"stem": "vertebrae_T7",              "tier": 1, "in_reg": True},
    36:  {"stem": "vertebrae_T8",              "tier": 1, "in_reg": True},
    37:  {"stem": "vertebrae_T9",              "tier": 1, "in_reg": True},
    38:  {"stem": "sacrum",                    "tier": 1, "in_reg": True},
    39:  {"stem": "vertebrae_S1",              "tier": 1, "in_reg": True},
    40:  {"stem": "spinal_cord",               "tier": 1, "in_reg": True},
    41:  {"stem": "hip_left",                  "tier": 1, "in_reg": True},
    42:  {"stem": "hip_right",                 "tier": 1, "in_reg": True},
    43:  {"stem": "femur_left",                "tier": 1, "in_reg": True},
    44:  {"stem": "femur_right",               "tier": 1, "in_reg": True},
    45:  {"stem": "sternum",                   "tier": 1, "in_reg": True},
    46:  {"stem": "costal_cartilages",         "tier": 1, "in_reg": True},
    47:  {"stem": "scapula_left",              "tier": 1, "in_reg": True},
    48:  {"stem": "scapula_right",             "tier": 1, "in_reg": True},
    49:  {"stem": "clavicula_left",            "tier": 1, "in_reg": True},
    50:  {"stem": "clavicula_right",           "tier": 1, "in_reg": True},
    60:  {"stem": "rib_left_1",               "tier": 1, "in_reg": True},
    61:  {"stem": "rib_left_2",               "tier": 1, "in_reg": True},
    62:  {"stem": "rib_left_3",               "tier": 1, "in_reg": True},
    63:  {"stem": "rib_left_4",               "tier": 1, "in_reg": True},
    64:  {"stem": "rib_left_5",               "tier": 1, "in_reg": True},
    65:  {"stem": "rib_left_6",               "tier": 1, "in_reg": True},
    66:  {"stem": "rib_left_7",               "tier": 1, "in_reg": True},
    67:  {"stem": "rib_left_8",               "tier": 1, "in_reg": True},
    68:  {"stem": "rib_left_9",               "tier": 1, "in_reg": True},
    69:  {"stem": "rib_left_10",              "tier": 1, "in_reg": True},
    70:  {"stem": "rib_left_11",              "tier": 1, "in_reg": True},
    71:  {"stem": "rib_left_12",              "tier": 1, "in_reg": True},
    72:  {"stem": "rib_right_1",              "tier": 1, "in_reg": True},
    73:  {"stem": "rib_right_2",              "tier": 1, "in_reg": True},
    74:  {"stem": "rib_right_3",              "tier": 1, "in_reg": True},
    75:  {"stem": "rib_right_4",              "tier": 1, "in_reg": True},
    76:  {"stem": "rib_right_5",              "tier": 1, "in_reg": True},
    77:  {"stem": "rib_right_6",              "tier": 1, "in_reg": True},
    78:  {"stem": "rib_right_7",              "tier": 1, "in_reg": True},
    79:  {"stem": "rib_right_8",              "tier": 1, "in_reg": True},
    80:  {"stem": "rib_right_9",              "tier": 1, "in_reg": True},
    81:  {"stem": "rib_right_10",             "tier": 1, "in_reg": True},
    82:  {"stem": "rib_right_11",             "tier": 1, "in_reg": True},
    83:  {"stem": "rib_right_12",             "tier": 1, "in_reg": True},
    100: {"stem": "stomach",                  "tier": 3, "in_reg": False},
    101: {"stem": "esophagus",               "tier": 3, "in_reg": False},
    102: {"stem": "urinary_bladder",         "tier": 3, "in_reg": False},
    103: {"stem": "prostate",                "tier": 3, "in_reg": False},
    104: {"stem": "adrenal_gland_left",      "tier": 3, "in_reg": False},
    105: {"stem": "adrenal_gland_right",     "tier": 3, "in_reg": False},
    106: {"stem": "kidney_cyst_left",        "tier": 3, "in_reg": False},
    107: {"stem": "kidney_cyst_right",       "tier": 3, "in_reg": False},
    108: {"stem": "gluteus_maximus_left",    "tier": 3, "in_reg": False},
    109: {"stem": "gluteus_maximus_right",   "tier": 3, "in_reg": False},
    110: {"stem": "gluteus_medius_left",     "tier": 3, "in_reg": False},
    111: {"stem": "gluteus_medius_right",    "tier": 3, "in_reg": False},
    112: {"stem": "gluteus_minimus_left",    "tier": 3, "in_reg": False},
    113: {"stem": "gluteus_minimus_right",   "tier": 3, "in_reg": False},
    114: {"stem": "lung_lower_lobe_left",    "tier": 3, "in_reg": False},
    115: {"stem": "lung_lower_lobe_right",   "tier": 3, "in_reg": False},
    116: {"stem": "lung_middle_lobe_right",  "tier": 3, "in_reg": False},
    117: {"stem": "lung_upper_lobe_left",    "tier": 3, "in_reg": False},
    118: {"stem": "lung_upper_lobe_right",   "tier": 3, "in_reg": False},
    119: {"stem": "heart",                   "tier": 3, "in_reg": False},
}


# ── I/O helpers ──────────────────────────────────────────────────────────────

def load_segmentation(path: str) -> np.ndarray:
    """Load a segmentation array from .nii / .nii.gz or .npy file."""
    p = Path(path)
    suffix = "".join(p.suffixes).lower()

    if suffix in (".nii", ".nii.gz"):
        try:
            import nibabel as nib
        except ImportError:
            raise ImportError("nibabel is required for NIfTI files. Install with: pip install nibabel")
        img = nib.load(str(p))
        return np.asarray(img.dataobj, dtype=np.int32)

    elif suffix == ".npy":
        return np.load(str(p)).astype(np.int32)

    else:
        raise ValueError(f"Unsupported file type: {suffix}. Expected .nii, .nii.gz, or .npy")


# ── Core logic ───────────────────────────────────────────────────────────────

def extract_labels(seg: np.ndarray) -> list[int]:
    """Return sorted list of unique non-zero integer labels present in the mask."""
    return sorted(int(v) for v in np.unique(seg) if v != 0)


def map_labels_to_organs(
    labels: list[int],
    tier: int | None = None,
    in_reg_only: bool = False,
) -> list[dict]:
    """
    Map label IDs to organ metadata.

    Parameters
    ----------
    labels      : list of integer label IDs found in the mask
    tier        : if set, keep only labels with this tier value
    in_reg_only : if True, keep only labels where in_reg is True

    Returns
    -------
    List of dicts with keys: label_id, stem, tier, in_reg, in_mask
    """
    results = []
    for label_id in labels:
        info = LABEL_MAP.get(label_id)
        if info is None:
            entry = {
                "label_id": label_id,
                "stem": "UNKNOWN",
                "tier": None,
                "in_reg": None,
                "in_mask": True,
            }
        else:
            entry = {
                "label_id": label_id,
                "stem": info["stem"],
                "tier": info["tier"],
                "in_reg": info["in_reg"],
                "in_mask": True,
            }

        # Optional filters
        if tier is not None and entry["tier"] != tier:
            continue
        if in_reg_only and not entry["in_reg"]:
            continue

        results.append(entry)

    return results


def voxel_counts(seg: np.ndarray, mapped: list[dict]) -> list[dict]:
    """Attach per-label voxel counts to the mapped results."""
    for entry in mapped:
        entry["voxel_count"] = int(np.sum(seg == entry["label_id"]))
    return mapped


def print_report(mapped: list[dict]) -> None:
    """Pretty-print the mapping results."""
    col_w = [10, 35, 6, 8, 12]
    header = f"{'Label ID':<{col_w[0]}}{'Organ (stem)':<{col_w[1]}}{'Tier':<{col_w[2]}}{'In Reg':<{col_w[3]}}{'Voxels':>{col_w[4]}}"
    sep = "-" * sum(col_w)
    print(sep)
    print(header)
    print(sep)
    for e in mapped:
        voxels = str(e.get("voxel_count", "n/a"))
        print(
            f"{e['label_id']:<{col_w[0]}}"
            f"{e['stem']:<{col_w[1]}}"
            f"{str(e['tier']):<{col_w[2]}}"
            f"{str(e['in_reg']):<{col_w[3]}}"
            f"{voxels:>{col_w[4]}}"
        )
    print(sep)
    print(f"Total organs found: {len(mapped)}\n")


# ── Programmatic API (import-friendly) ───────────────────────────────────────

def process_segmentation(
    seg_path: str,
    tier: int | None = None,
    in_reg_only: bool = False,
    count_voxels: bool = True,
) -> list[dict]:
    """
    High-level function: load file → extract labels → map to organs.

    Example
    -------
    >>> results = process_segmentation("mask.nii.gz", tier=2)
    >>> for r in results:
    ...     print(r["label_id"], r["stem"], r["voxel_count"])
    """
    seg = load_segmentation(seg_path)
    labels = extract_labels(seg)
    mapped = map_labels_to_organs(labels, tier=tier, in_reg_only=in_reg_only)
    if count_voxels:
        mapped = voxel_counts(seg, mapped)
    return mapped






In [3]:
results = process_segmentation(
    seg_path="../ncct_cect/vindr_ds/ts_segmentation/1.2.840.113619.2.278.3.717616.260.1578649418.868_1.2.840.113619.2.278.3.717616.260.1578649419.700.11_seg_full.nii.gz",
    tier=None,        
    in_reg_only=False,
    count_voxels=True,
)

print_report(results)

ValueError: Unsupported file type: .2.840.113619.2.278.3.717616.260.1578649418.868_1.2.840.113619.2.278.3.717616.260.1578649419.700.11_seg_full.nii.gz. Expected .nii, .nii.gz, or .npy

In [15]:
import nibabel as nib
import numpy as np

seg = nib.load("../ncct_cect/vindr_ds/ts_segmentation/1.2.840.113619.2.278.3.717616.260.1578649418.868_1.2.840.113619.2.278.3.717616.260.1578649418.873.3_seg_full.nii.gz")
data = seg.get_fdata().astype(np.int32)

print("Shape:", data.shape)
print("Unique labels:", np.unique(data))

Shape: (267, 267, 334)
Unique labels: [  0   1   2   3   4   5   7   8   9  10  13  14  15  16  17  18  19  21
  22  23  24  25  26  27  28  32  33  34  35  36  37  38  39  40  41  42
  43  44  45  46  47  48  63  64  65  66  67  68  69  70  71  75  76  77
  78  79  80  81  82  83 100 101 102 103 104 105 108 109 110 111 112 113
 114 115 116 117 118 119]


In [16]:
labels = [int(l) for l in np.unique(data) if l != 0]

for l in labels:
    info = LABEL_MAP.get(l)
    name = info["stem"] if info else "UNKNOWN"
    print(f"{l:>4}  {name}")

   1  liver
   2  spleen
   3  kidney_left
   4  kidney_right
   5  pancreas
   7  autochthon_left
   8  autochthon_right
   9  iliopsoas_left
  10  iliopsoas_right
  13  aorta
  14  inferior_vena_cava
  15  portal_vein_and_splenic_vein
  16  iliac_artery_left
  17  iliac_artery_right
  18  iliac_vena_left
  19  iliac_vena_right
  21  vertebrae_L1
  22  vertebrae_L2
  23  vertebrae_L3
  24  vertebrae_L4
  25  vertebrae_L5
  26  vertebrae_T10
  27  vertebrae_T11
  28  vertebrae_T12
  32  vertebrae_T4
  33  vertebrae_T5
  34  vertebrae_T6
  35  vertebrae_T7
  36  vertebrae_T8
  37  vertebrae_T9
  38  sacrum
  39  vertebrae_S1
  40  spinal_cord
  41  hip_left
  42  hip_right
  43  femur_left
  44  femur_right
  45  sternum
  46  costal_cartilages
  47  scapula_left
  48  scapula_right
  63  rib_left_4
  64  rib_left_5
  65  rib_left_6
  66  rib_left_7
  67  rib_left_8
  68  rib_left_9
  69  rib_left_10
  70  rib_left_11
  71  rib_left_12
  75  rib_right_4
  76  rib_right_5
  77  rib_right

In [17]:
for l in labels:
    info = LABEL_MAP.get(l)
    name = info["stem"] if info else "UNKNOWN"
    count = int(np.sum(data == l))
    print(f"{l:>4}  {name:<40}  {count:>10} voxels")

   1  liver                                         482556 voxels
   2  spleen                                        234012 voxels
   3  kidney_left                                    46532 voxels
   4  kidney_right                                   47851 voxels
   5  pancreas                                       11972 voxels
   7  autochthon_left                                91011 voxels
   8  autochthon_right                               84521 voxels
   9  iliopsoas_left                                 47612 voxels
  10  iliopsoas_right                                47214 voxels
  13  aorta                                          52730 voxels
  14  inferior_vena_cava                             12096 voxels
  15  portal_vein_and_splenic_vein                    6225 voxels
  16  iliac_artery_left                               4251 voxels
  17  iliac_artery_right                              3919 voxels
  18  iliac_vena_left                                 4911 voxels
  19  ilia